# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

# 1. My Lane and Why

I'm choosing Lane 1: Ranking Signal Analysis.

Before I build any model or scoring system, I want to know which observable
signals in the starter dataset — things like `avg_position`, `ctr`,
`freshness_tier`, `word_count`, `days_since_last_update`, and `trend_direction`
— actually move together with visibility and engagement outcomes
(`impressions_90d`, `clicks_90d`, `engagement_rate`), and which ones don't.
Right now I don't know if the signals I'd naturally reach for (like word
count or content age) are the ones that matter, or if I'd be optimizing the
wrong lever for the next 7 weeks.

I picked this lane over the others because it's the most foundational: it
tells me what's actually happening in the data before I commit to a
specific downstream task (like building a refresh-priority model or a
clustering system). If I skip this step and jump straight to modeling, I risk
building something technically working but pointed at a signal nobody should
care about.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

# 2. The Question

Research question: Which content and search signals in the starter
dataset — such as `avg_position`, `ctr`, `freshness_tier`,
`days_since_last_update`, `word_count_tier`, and `trend_direction` — are
associated with a page's visibility (`impressions_90d`), clicks
(`clicks_90d`), or engagement (`engagement_rate`)? And how strong is that
association?

Unit of analysis: one content page (`content_id`), measured over the
90-day window in the dataset (with `impressions_last_30d` /
`impressions_prev_30d` giving a shorter-term trend view via `trend_direction`
and `trend_pct`).

Output: a ranked list of signals with their correlation/effect size
against visibility and engagement outcomes, plus 3-5 charts showing the
strongest patterns — e.g. does `avg_position` correlate with `ctr` the way
we'd expect, do fresher pages (`freshness_tier`) show better
`engagement_rate`, does `trend_direction == "down"` cluster with any
particular `word_count_tier` or `age_tier`.

Decision this improves: which signal(s) a content team should actually
watch when deciding which pages to review, refresh, or prioritize — instead
of reviewing pages in whatever order they were published, or by age alone.

Action someone takes: a content strategist or SEO lead uses this report
to decide which signal to act on first when triaging a backlog of pages —
for example, prioritizing pages with `trend_direction == "down"` and high
`impressions_prev_30d` (real lost visibility) over pages that are simply in
an older `age_tier` but stable.

Cost of a wrong call: if I claim a signal like `word_count` or
`age_tier` matters when the real driver is something else (or a confound
like seasonality or a site-wide event affecting `impressions_last_30d`
across many pages at once), the team spends review time on the wrong pages
while pages genuinely losing visibility (real `trend_pct` decline) keep
declining. A false signal here isn't harmless — it's wasted attention on a
scarce resource (content team time).

Why data/ML helps at all: with ~30k pages and dozens of signal columns,
no one can eyeball which of `avg_position`, `ctr`, `freshness_tier`,
`word_count_tier`, or the trend fields actually move with
`impressions_90d` or `engagement_rate`, and which are just noise. Simple
correlation analysis and effect sizes across the whole dataset can surface
patterns a human skimming spreadsheets would miss or mis-rank by intuition
alone.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [3]:
import os

REPO_NAME = "FlyRankAI_ML-Engineering_Internship"
REPO_URL = "https://github.com/Sadman-Islam-Chowdhury-Samin/FlyRankAI_ML-Engineering_Internship.git"

# If we're not already inside the repo, clone it and cd in
if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}
os.chdir(REPO_NAME)

print("Working directory:", os.getcwd())
print(os.listdir("data/raw"))


Working directory: /content/FlyRankAI_ML-Engineering_Internship/FlyRankAI_ML-Engineering_Internship
['content_refresh_anonymized.csv']


In [4]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Total pages:", len(df))
print("Share of pages with trend_direction == 'down':", (df["trend_direction"] == "down").mean())
print("Median impressions_90d:", df["impressions_90d"].median())
print("Median avg_position:", df["avg_position"].median())
print("Correlation: avg_position vs ctr:", df["avg_position"].corr(df["ctr"]))
print("Mean engagement_rate by freshness_tier:")
print(df.groupby("freshness_tier")["engagement_rate"].mean())
print("Share of pages with engagement_rate > 0, by freshness_tier:")
print(df.groupby("freshness_tier")["engagement_rate"].apply(lambda x: (x > 0).mean()))

Total pages: 30000
Share of pages with trend_direction == 'down': 0.5420666666666667
Median impressions_90d: 731.0
Median avg_position: 10.8
Correlation: avg_position vs ctr: -0.07259029976832464
Mean engagement_rate by freshness_tier:
freshness_tier
0-30      2.599727
181+      2.028333
31-90     2.134971
91-180    2.406133
Name: engagement_rate, dtype: float64
Share of pages with engagement_rate > 0, by freshness_tier:
freshness_tier
0-30      0.239795
181+      0.080460
31-90     0.234286
91-180    0.371279
Name: engagement_rate, dtype: float64


Across 30,000 pages, 54.2% are currently trending down (`trend_direction ==
"down"`), and `avg_position` shows only a weak correlation with `ctr`
(r ≈ -0.073) — a real but non-obvious pattern. Engagement also varies
unevenly by `freshness_tier` rather than in a clean line. These numbers show
a large, measurable problem (more than half of pages losing visibility) with
signals that don't behave simply or predictably — exactly the kind of
pattern Lane 1 is meant to investigate over the next 7 weeks.

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*
## 4. Careful Words: What I Can and Can't Claim

**What this analysis can say:**
- These are **observed associations** in a 30,000-page anonymized sample, not
  universal laws. For example, `avg_position` and `ctr` show a weak negative
  correlation (r ≈ -0.073) — directionally consistent with "better position,
  somewhat better CTR," but far too weak to be a strong predictive rule on
  its own.
- Patterns are **directional signals**, not fixed rates. `engagement_rate`
  by `freshness_tier` doesn't move in a clean straight line — pages aged
  91-180 days show the highest share of engaged sessions (37%), while 181+
  days shows the lowest (8%), with 0-30 and 31-90 landing in between at a
  similar level. This suggests freshness alone doesn't explain engagement
  in a simple way, and other factors (content type, intent, word count)
  likely interact with it.
- These findings are **decision-support**, not a decision itself. They can
  help a content strategist decide which pages or signals deserve a closer
  look first, narrowing 30,000 pages down to a shorter, evidence-backed
  shortlist for human review.
- With 54.2% of pages currently trending down, the analysis can honestly say
  a large share of this content base is losing visibility — a real,
  measurable problem worth investigating further, not an assumption.

**What this analysis can never claim:**
- **Not causal proof.** A correlation between `avg_position` and `ctr`, or
  between `freshness_tier` and `engagement_rate`, does not mean one causes
  the other. Confounding factors (content type, search intent, seasonality,
  a client's overall site changes) could explain some or all of these
  patterns.
- **Not "predicting Google."** This is not a model of Google's ranking
  algorithm, and no result here should be read as "this is a ranking
  factor." It's an analysis of *this client's observed outcomes* alongside
  *this client's observable signals* — nothing more.
- **Not a guarantee for any single page.** Aggregate patterns across 30,000
  pages don't guarantee that acting on any one signal will improve any one
  specific page's performance.
- **Not final without human judgment.** These numbers are a starting point
  for a content team's review process, not a replacement for it.

## Self-check

Before you submit, confirm each line honestly:

- [✅ ] Every section above is filled — markdown thinking AND the code that backs it
- [✅ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅ ] No client names, URLs, or private queries anywhere
- [✅ ] My claims use careful words: observed, measured, directional, decision-support
- [✅ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.